# Challenge 2: House Prices - Advanced Regression Techniques

## Huấn luyện mô hình cơ bản (Baseline Model Training)

Các mô hình được huấn luyện trên tập dữ liệu gốc (chưa qua feature engineering), nhằm đánh giá hiệu năng cơ bản ban đầu.

### Khai báo thư viện

In [1]:
import pandas as pd
import numpy as np
import random
import os, sys
from IPython import display


from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

### Tham số thực nghiệm

In [11]:
params = {}

# Thư mục thí nghiệm
params["exps_dir"]  = "../exps"
params["exp_name"]  = "challenge2_houseprice_standard"
params["save_dir"]  = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

# Đường dẫn dữ liệu đã preprocess (before FE)
params["data_path"] = f'{params["exps_dir"]}/data/train.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

os.makedirs(params["save_dir"], exist_ok=True)

random.seed(params["random_state"])
np.random.seed(params["random_state"])
os.environ["PYTHONHASHSEED"] = str(params["random_state"])

print("Save dir:", params["save_dir"])
print("Train path:", params["data_path"])
print("Test path :", params["test_path"])


Save dir: ../exps/result1_challenge2_houseprice_standard
Train path: ../exps/data/train.xlsx
Test path : ../exps/data/test.xlsx


### Nạp dữ liệu

In [15]:
df_train = pd.read_excel(params["data_path"])
df_test  = pd.read_excel(params["test_path"])

In [16]:
df_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SalePrice
0,0.073375,-0.231877,-0.207142,0.651479,-0.517200,1.050994,0.878668,0.514104,0.575425,-0.288653,...,0,0,1,0,0,0,0,1,0,208500
1,-0.872563,0.437043,-0.091886,-0.071836,2.179628,0.156734,-0.429577,-0.570750,1.171992,-0.288653,...,0,0,1,0,0,0,0,1,0,181500
2,0.073375,-0.098093,0.073480,0.651479,-0.517200,0.984752,0.830215,0.325915,0.092907,-0.288653,...,0,0,1,0,0,0,0,1,0,223500
3,0.309859,-0.454850,-0.096897,0.651479,-0.517200,-1.863632,-0.720298,-0.570750,-0.499274,-0.288653,...,0,0,1,1,0,0,0,0,0,140000
4,0.073375,0.615421,0.375148,1.374795,-0.517200,0.951632,0.733308,1.366489,0.463568,-0.288653,...,0,0,1,0,0,0,0,1,0,250000


In [17]:
df_test.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,-0.872563,0.437043,0.110763,-0.795151,0.381743,-0.340077,-1.156380,-0.570750,0.053428,0.604293,...,0,0,0,1,0,0,0,0,1,0
1,-0.872563,0.481637,0.375850,-0.071836,0.381743,-0.439440,-1.301740,0.027027,1.051363,-0.288653,...,0,0,0,1,0,0,0,0,1,0
2,0.073375,0.169475,0.332053,-0.795151,-0.517200,0.852269,0.636400,-0.570750,0.761852,-0.288653,...,0,0,0,1,0,0,0,0,1,0
3,0.073375,0.347853,-0.054002,-0.071836,0.381743,0.885390,0.636400,-0.460051,0.347326,-0.288653,...,0,0,0,1,0,0,0,0,1,0
4,1.492282,-1.212959,-0.552407,1.374795,-0.517200,0.686666,0.345679,-0.570750,-0.396190,-0.288653,...,0,0,0,1,0,0,0,0,1,0


In [19]:
# Tách X, y
y = df_train["SalePrice"]
X = df_train.drop(columns=["SalePrice"])


print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (1460, 203)
y shape: (1460,)
Train X: (1168, 203)
Valid X: (292, 203)
Train y: (1168,)
Valid y: (292,)


In [20]:
kfold = KFold(
    n_splits=params["k_fold"],
    shuffle=True,
    random_state=params["random_state"]
)

print(f"Total rows in X_train: {len(X_train)}\n")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"---- Fold {fold} ----")
    print(f"Train size: {len(train_idx)}")
    print(f"Valid size: {len(valid_idx)}")
    print(f"Train idx sample: {train_idx[:10]}")
    print(f"Valid idx sample: {valid_idx[:10]}")
    print()

Total rows in X_train: 1168

---- Fold 0 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [ 23  44  49  51  54  58  70  86 101 107]

---- Fold 1 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [10 31 43 56 59 63 76 83 88 96]

---- Fold 2 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  4  5  7  8  9 10 11 13]
Valid idx sample: [ 2  3  6 12 25 27 30 39 47 55]

---- Fold 3 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  2  3  4  6  7  8  9 10]
Valid idx sample: [ 5 29 33 60 65 71 77 82 84 92]

---- Fold 4 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 1  2  3  4  5  6  8 10 11 12]
Valid idx sample: [  0   7   9  62  69  79  81  90  97 104]

---- Fold 5 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [11 15 18 24 28 41 42 61 73 74]

---- Fold 6 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1 

### Lựa chọn mô hình mặc định

In [21]:
models = [
    ("Extra Trees", ExtraTreesRegressor(random_state=params["random_state"])),
    ("Random Forest", RandomForestRegressor(random_state=params["random_state"])),
    ("LightGBM", LGBMRegressor(
        random_state=params["random_state"],
        n_estimators=2000,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        verbose=-1
    )),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=params["random_state"])),
    ("XGBoost", XGBRegressor(
        random_state=params["random_state"],
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror"
    ))
]

### Huấn luyện và đánh giá từng mô hình

In [25]:
results = []
baseline_results = {}

y_log = np.log1p(y)

for name, model in models:
    print(f"===== Model: {name} =====")

    baseline_results[name] = {"mae": [], "rmse": [], "r2": []}

    kf = KFold(
        n_splits=params["k_fold"],
        shuffle=True,
        random_state=params["random_state"]
    )

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y_log)):
        X_tr, y_tr = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[valid_idx], y_log.iloc[valid_idx]

        # Train
        model.fit(X_tr, y_tr)

        # Predict (log-space)
        y_pred = model.predict(X_val)

        # Convert back for metrics
        y_pred_real = np.expm1(y_pred)
        y_val_real  = np.expm1(y_val)

        mae = mean_absolute_error(y_val_real, y_pred_real)
        rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
        r2 = r2_score(y_val_real, y_pred_real)

        baseline_results[name]["mae"].append(mae)
        baseline_results[name]["rmse"].append(rmse)
        baseline_results[name]["r2"].append(r2)

        print(f" Fold {fold}: RMSE = {rmse:.4f}")

    print(f"--> Mean RMSE: {np.mean(baseline_results[name]['rmse']):.4f} ± {np.std(baseline_results[name]['rmse']):.4f}\n")

    results.append([
        name,
        np.mean(baseline_results[name]["mae"]),
        np.mean(baseline_results[name]["rmse"]),
        np.mean(baseline_results[name]["r2"])
    ])

===== Model: Extra Trees =====
 Fold 0: RMSE = 29915.5827
 Fold 1: RMSE = 25939.0226
 Fold 2: RMSE = 23241.1181
 Fold 3: RMSE = 31106.4570
 Fold 4: RMSE = 39200.5738
 Fold 5: RMSE = 28022.0860
 Fold 6: RMSE = 28740.9396
 Fold 7: RMSE = 30494.8746
 Fold 8: RMSE = 23607.4543
 Fold 9: RMSE = 22392.0528
--> Mean RMSE: 28266.0162 ± 4716.7439

===== Model: Random Forest =====
 Fold 0: RMSE = 31622.6274
 Fold 1: RMSE = 27145.6107
 Fold 2: RMSE = 19686.5201
 Fold 3: RMSE = 30246.4474
 Fold 4: RMSE = 41194.1173
 Fold 5: RMSE = 32013.1920
 Fold 6: RMSE = 32289.4032
 Fold 7: RMSE = 26352.3087
 Fold 8: RMSE = 25727.3712
 Fold 9: RMSE = 21279.6906
--> Mean RMSE: 28755.7289 ± 5864.5915

===== Model: LightGBM =====
 Fold 0: RMSE = 29645.1754
 Fold 1: RMSE = 25976.9218
 Fold 2: RMSE = 22577.4426
 Fold 3: RMSE = 31975.3268
 Fold 4: RMSE = 31120.0842
 Fold 5: RMSE = 26607.8294
 Fold 6: RMSE = 32311.1098
 Fold 7: RMSE = 22900.4023
 Fold 8: RMSE = 22185.7517
 Fold 9: RMSE = 16742.1453
--> Mean RMSE: 26204

In [26]:
df_results = pd.DataFrame(
    results,
    columns=["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
)

# Sắp xếp theo RMSE thấp nhất (mục tiêu RMSE càng thấp càng tốt)
df_results = df_results.sort_values(by="Mean_RMSE", ascending=True)

display.display(df_results)

,Model,Mean_MAE,Mean_RMSE,Mean_R2
4,XGBoost,14369.802970,24646.901503,0.894945
2,LightGBM,15706.102580,26204.218922,0.887340
3,Gradient Boosting,15728.859034,26863.640912,0.873192
0,Extra Trees,16911.541008,28266.016159,0.864837
1,Random Forest,17259.236722,28755.728864,0.859149


**Nhận xét kết quả thực nghiệm các mô hình**

- XGBoost đạt hiệu suất tốt nhất với RMSE = 24,647 và R² = 0.895, vượt trội so với các mô hình còn lại -> XGBoost đang khai thác tốt cấu trúc dữ liệu, học được các quan hệ phi tuyến và tương tác giữa các thuộc tính. Đây là mô hình phù hợp nhất trong nhóm baseline hiện tại.

- LightGBM xếp thứ hai với RMSE = 26,204, vẫn rất cạnh tranh nhưng kém hơn XGBoost đáng kể. Đây là mô hình mạnh, đặc biệt khi xử lý dữ liệu nhiều chiều, và có thể cải thiện thêm khi tinh chỉnh hyperparameters.

- Gradient Boosting đạt RMSE = 26,864, khá gần với LightGBM, nhưng thấp hơn cả về độ chính xác lẫn độ ổn định. Điều này phản ánh đúng đặc tính: GradientBoostingRegressor mặc định thường yếu hơn LightGBM/XGBoost khi chưa tối ưu.

- Extra Trees (RMSE = 28,266) và Random Forest (RMSE = 28,756) đều cho kết quả thấp hơn đáng kể. Đây là điều thường thấy vì các mô hình bagging khó cạnh tranh với boosting trong các bài toán hồi quy phi tuyến như Ames Housing.

=> XGBoost và LightGBM là 2 mô hình có hiệu suất cao nhất và ổn định nhất trong thực nghiệm hiện tại.